# Patient Health Records: Data Preprocessing & Feature Engineering
**College Project | Python, Pandas, NumPy, Scikit-learn and Matplotlib**

## Objective
Clean a synthetic healthcare dataset by studying missing values, comparing six imputation approaches, detecting outliers with three detection methods, applying Winsorization, and preparing a machine-learning-ready dataset.

> **Important:** This is a synthetic educational dataset. It contains no real patient information and must not be used for clinical decisions.

## Dataset dictionary

| Field | Type | Description |
|---|---|---|
| `patient_id` | Identifier | Unique patient record |
| `age` | Numeric | Age in years |
| `gender` | Categorical | Male / Female |
| `region` | Categorical | North / South / East / West |
| `bmi` | Numeric | Body Mass Index |
| `blood_pressure` | Numeric | Average systolic pressure (mmHg) |
| `cholesterol` | Numeric | Cholesterol (mg/dL) |
| `glucose` | Numeric | Fasting glucose (mg/dL) |
| `disease_risk` | Binary target | 0 = Low Risk, 1 = High Risk |

The raw dataset intentionally includes missing values and synthetic extreme measurements.

## 1. Import libraries and load data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # enables IterativeImputer
from sklearn.impute import IterativeImputer

DATA_PATH = Path("../data/patient_health_records_raw.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/patient_health_records_raw.csv")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Dataset overview and missing-value report

In [ ]:
print(df.info())
display(df.describe(include="all").T)

missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percentage", ascending=False)
display(missing_report)

ax = missing_report["missing_percentage"].sort_values().plot(
    kind="barh", figsize=(9, 5), title="Missing Values by Column (%)"
)
ax.set_xlabel("Missing values (%)")
plt.tight_layout()
plt.show()

**Interpretation:** The table and chart identify which fields need imputation. `patient_id` and `disease_risk` are expected to remain complete. The target is not imputed because changing target labels would alter the supervised-learning outcome.

## 3. Compare missing-value imputation strategies

In [ ]:
numeric_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]
categorical_cols = ["gender", "region"]
feature_cols = numeric_cols + categorical_cols

# A. Simple imputation: numerical median (robust to skew/outliers)
simple_num = SimpleImputer(strategy="median")
simple_cat = SimpleImputer(strategy="most_frequent")
simple_df = df.copy()
simple_df[numeric_cols] = simple_num.fit_transform(simple_df[numeric_cols])
simple_df[categorical_cols] = simple_cat.fit_transform(simple_df[categorical_cols])

# B. Most-frequent imputation for categorical fields
mode_df = df.copy()
mode_df[categorical_cols] = SimpleImputer(strategy="most_frequent").fit_transform(
    mode_df[categorical_cols]
)

print("Simple imputation missing counts:")
display(simple_df[feature_cols].isna().sum().to_frame("remaining_missing"))
print("Most-frequent categorical values:")
display(df[categorical_cols].mode().iloc[0].to_frame("most_frequent_value"))

In [ ]:
# C. Missing indicator + random-sample imputation
random_sample_df = df.copy()
missing_flags = pd.DataFrame(index=df.index)
for col in feature_cols:
    missing_flags[col + "_was_missing"] = random_sample_df[col].isna().astype(int)
    miss = random_sample_df[col].isna()
    observed = random_sample_df.loc[~miss, col].dropna()
    if miss.any() and len(observed):
        random_sample_df.loc[miss, col] = np.random.default_rng(42).choice(
            observed.to_numpy(), size=miss.sum(), replace=True
        )
random_sample_df = pd.concat([random_sample_df, missing_flags], axis=1)

# D. KNN imputation for numeric variables; categorical variables use mode
knn_df = df.copy()
knn_df[numeric_cols] = KNNImputer(n_neighbors=5, weights="distance").fit_transform(
    knn_df[numeric_cols]
)
knn_df[categorical_cols] = SimpleImputer(strategy="most_frequent").fit_transform(
    knn_df[categorical_cols]
)

# E. MICE-style chained equations using IterativeImputer
mice_df = df.copy()
mice_df[numeric_cols] = IterativeImputer(
    max_iter=10, random_state=42
).fit_transform(mice_df[numeric_cols])
mice_df[categorical_cols] = SimpleImputer(strategy="most_frequent").fit_transform(
    mice_df[categorical_cols]
)

imputation_comparison = pd.DataFrame({
    "Method": ["Median + mode", "Most frequent (categorical)",
               "Missing indicator + random sample", "KNN + categorical mode",
               "MICE / IterativeImputer"],
    "Remaining missing feature cells": [
        int(simple_df[feature_cols].isna().sum().sum()),
        int(mode_df[feature_cols].isna().sum().sum()),
        int(random_sample_df[feature_cols].isna().sum().sum()),
        int(knn_df[feature_cols].isna().sum().sum()),
        int(mice_df[feature_cols].isna().sum().sum())
    ],
    "Preserves missingness flag": ["No", "No", "Yes", "No", "No"]
})
display(imputation_comparison)

### Imputation method notes
- **Mean/median SimpleImputer:** fills numeric gaps; median is less sensitive to extreme values.
- **Most-frequent:** fills categorical gaps using the mode.
- **Missing indicator + random sample:** retains a flag showing where values were missing and samples from observed values.
- **KNNImputer:** estimates missing numeric values from similar rows.
- **MICE / IterativeImputer:** estimates each incomplete numeric feature from the other numeric features in chained rounds.

For this assignment, categorical fields are mode-imputed for KNN and MICE workflows. The target `disease_risk` is never included in imputation.

## 4. Outlier detection: Z-score, IQR and percentile methods

In [ ]:
# Use the median-imputed dataset as a common baseline for comparisons
outlier_base = simple_df.copy()
outlier_cols = ["bmi", "blood_pressure", "cholesterol", "glucose"]

# Z-score: |z| > 3
zscore_flags = pd.DataFrame(index=outlier_base.index)
for col in outlier_cols:
    sd = outlier_base[col].std(ddof=0)
    z = (outlier_base[col] - outlier_base[col].mean()) / sd if sd else 0
    zscore_flags[col] = z.abs() > 3

# IQR: outside Q1 - 1.5*IQR and Q3 + 1.5*IQR
iqr_flags = pd.DataFrame(index=outlier_base.index)
iqr_bounds = {}
for col in outlier_cols:
    q1, q3 = outlier_base[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    iqr_bounds[col] = (lower, upper)
    iqr_flags[col] = (outlier_base[col] < lower) | (outlier_base[col] > upper)

# Percentile: below 1st or above 99th percentile
percentile_flags = pd.DataFrame(index=outlier_base.index)
percentile_bounds = {}
for col in outlier_cols:
    lower, upper = outlier_base[col].quantile([0.01, 0.99])
    percentile_bounds[col] = (lower, upper)
    percentile_flags[col] = (outlier_base[col] < lower) | (outlier_base[col] > upper)

outlier_report = pd.DataFrame({
    "Z-score count": zscore_flags.sum(),
    "IQR count": iqr_flags.sum(),
    "Percentile count": percentile_flags.sum()
})
display(outlier_report)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, ax in zip(outlier_cols, axes.ravel()):
    ax.boxplot(outlier_base[col].dropna(), vert=True)
    ax.set_title(f"{col} — before treatment")
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

**Interpretation:** Z-score flags values more than three standard deviations from the mean. IQR flags values outside the 1.5×IQR fences. The percentile method flags the tails below the 1st and above the 99th percentile. Counts can differ because the methods use different definitions.

## 5. Remove outliers and apply Winsorization

In [ ]:
# Row-removal comparison: remove rows flagged by IQR in any selected measurement
iqr_any_outlier = iqr_flags.any(axis=1)
removed_outlier_df = outlier_base.loc[~iqr_any_outlier].copy()

# Winsorization: cap each measurement at its 1st and 99th percentile
winsor_df = outlier_base.copy()
for col in outlier_cols:
    lo, hi = winsor_df[col].quantile([0.01, 0.99])
    winsor_df[col] = winsor_df[col].clip(lower=lo, upper=hi)

comparison = pd.DataFrame({
    "Dataset": ["Before treatment", "After IQR row removal", "After Winsorization"],
    "Rows": [len(outlier_base), len(removed_outlier_df), len(winsor_df)],
    "Columns": [outlier_base.shape[1], removed_outlier_df.shape[1], winsor_df.shape[1]]
})
display(comparison)

summary_before = outlier_base[outlier_cols].describe().T
summary_after = winsor_df[outlier_cols].describe().T
display(summary_before.add_prefix("before_").join(summary_after.add_prefix("after_")))

**Interpretation:** IQR row removal reduces the number of records and may discard valid extreme patients. Winsorization retains every row while limiting the influence of extreme values. Review the domain context before treating any clinical measurement as an error.

## 6. Build the final clean dataset

In [ ]:
# Final strategy: median numeric imputation, mode categorical imputation,
# missingness indicators for originally incomplete features, and Winsorization.
final_df = df.copy()

for col in feature_cols:
    final_df[col + "_was_missing"] = final_df[col].isna().astype(int)

final_df[numeric_cols] = SimpleImputer(strategy="median").fit_transform(final_df[numeric_cols])
final_df[categorical_cols] = SimpleImputer(strategy="most_frequent").fit_transform(final_df[categorical_cols])

for col in outlier_cols:
    lo, hi = final_df[col].quantile([0.01, 0.99])
    final_df[col] = final_df[col].clip(lower=lo, upper=hi)

# Keep ID and target intact
assert final_df["patient_id"].equals(df["patient_id"])
assert final_df["disease_risk"].equals(df["disease_risk"])
assert final_df[feature_cols].isna().sum().sum() == 0

Path("outputs").mkdir(exist_ok=True)
final_df.to_csv("outputs/patient_health_records_cleaned.csv", index=False)
print("Final dataset shape:", final_df.shape)
print("Remaining missing feature values:", final_df[feature_cols].isna().sum().sum())
display(final_df.head())

## 7. Final quality checks and visual comparison

In [ ]:
quality_report = pd.DataFrame({
    "Before missing cells": df.isna().sum(),
    "After missing cells": final_df.isna().sum()
})
display(quality_report)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[outlier_cols].boxplot(ax=axes[0])
axes[0].set_title("Before outlier treatment")
final_df[outlier_cols].boxplot(ax=axes[1])
axes[1].set_title("After Winsorization")
plt.tight_layout()
plt.show()

print("Target distribution (unchanged):")
display(final_df["disease_risk"].value_counts().rename_axis("disease_risk").to_frame("records"))

## 8. Conclusion

The workflow identifies missing data, compares simple, random-sample, KNN and MICE-style imputation, and evaluates three outlier detection methods. The final dataset uses median imputation for numeric fields, most-frequent imputation for categorical fields, missingness indicators, and percentile Winsorization. This preserves all patient rows and leaves `patient_id` and `disease_risk` unchanged.

**Limitations:** The dataset is synthetic. Imputation and outlier thresholds are teaching choices, not clinical recommendations. For real machine learning, fit preprocessing only on the training set (preferably in a scikit-learn Pipeline) to prevent data leakage.